In [1]:
# bowaka_v2_lab notebook bootstrap cell — DO NOT EDIT BY HAND.
# Adds the lab's src/ (and its bowaka_common dependency) to sys.path and pins
# the working directory to the repo root, so `import bowaka_v2_lab` and
# repo-root-relative CONFIG_PATH parameters resolve identically under jupyter,
# papermill, and the QuantsLab scheduler.
import os
import sys
from pathlib import Path

_lab_root = None
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "bowaka_v2_lab" / "__init__.py").is_file():
        _lab_root = _candidate
        break
if _lab_root is None:
    raise RuntimeError(
        f"bowaka_v2_lab bootstrap: src/bowaka_v2_lab/ not found at or above {Path.cwd()}"
    )

# Pin CWD to the repo root so repo-root-relative CONFIG_PATH values resolve
# regardless of how the notebook was launched (jupyter CWD = notebook dir,
# scheduler = repo root). The lab ALWAYS lives at
# ``<repo_root>/research_notebooks/bowaka_v2_lab``, so derive the repo root from
# that canonical layout. A marker-only heuristic ("a dir with research_notebooks/
# AND Makefile") mis-matches the lab dir itself when it carries a Makefile and a
# stray nested research_notebooks/ — which then chdir's one level too deep and
# breaks every repo-root-relative path.
if _lab_root.parent.name == "research_notebooks":
    _repo_root = _lab_root.parent.parent
else:
    # Fallback: the repo root holds research_notebooks/bowaka_common (the sibling
    # package) — a marker a stray nested research_notebooks/ inside the lab lacks.
    _repo_root = _lab_root
    for _candidate in [_lab_root, *_lab_root.parents]:
        if (_candidate / "research_notebooks" / "bowaka_common").is_dir():
            _repo_root = _candidate
            break
os.chdir(_repo_root)

# Make the lab and its bowaka_common dependency importable from the working
# tree, even when the packages are not pip-installed. v1 bowaka_lab is
# deliberately excluded — v2 must not import v1.
for _src in (_lab_root / "src",
             _repo_root / "research_notebooks" / "bowaka_common" / "src"):
    if _src.is_dir() and str(_src) not in sys.path:
        sys.path.insert(0, str(_src))

import bowaka_v2_lab  # noqa: F401
print(f"bowaka_v2_lab {bowaka_v2_lab.__version__} (cwd={_repo_root})")


bowaka_v2_lab 0.1.0 (cwd=/quants-lab)


In [ ]:
# Papermill parameters.
# NOTE: the prior bowaka_v2_walkforward_optuna.yml is QUARANTINED (realism
# audit 2026-05-22 §P0-001). The default below points at the Phase-8
# walk-forward-purpose contract-parity config that matches today's lake.
#
# MODE NOTE: FEED='auto' resolves to SIP (bars+quotes present). The feed->mode
# rule would upgrade simulation.mode to intended_realism, but IR is INFEASIBLE on
# this lake — NOT for lack of halt data (that's now backfilled to statuses/) but
# because the microcap minute-bar density can't support a faithful tape replay
# (the coverage_missing_late_session DQ gate, ~9.5% > 1%). So the SEARCH stays
# current_code_parity (thorough + cheap), and realistic execution is assessed by
# RE-SCORING the top finalists under fast_realism (honest depth/participation
# fills) — the two-stage §10i method (search cheap, validate finalists honestly).
# See RESCORE_MODE + the Top-N sweep cell below.
#
# Speedup report v2 §4 P2 / Phase 2 — default points at the workstation
# overlay (192 GiB / 18-core box). The base optuna config carries the more
# conservative reserve_system_gib=32 / strict_parallel=false defaults; flip
# CONFIG_PATH below if you intentionally want those.
CONFIG_PATH = 'research_notebooks/bowaka_v2_lab/configs/_local_container_matrix.yml'
N_TRIALS = 5000          # None -> optuna.n_trials from the config; or set an integer
N_STARTUP_TRIALS = 500  # None -> optuna.n_startup_trials; random trials before TPE
N_JOBS = None            # None -> optuna.n_jobs (Phase 5 default: 8 workers on PostgreSQL)
FEED = 'auto'            # 'auto' (SIP > IEX > synthetic) | 'sip' | 'iex' | 'synthetic'
MODE_OVERRIDE = 'current_code_parity'  # SEARCH mode; pins sim mode over the
                         # feed-derived one (None -> feed-derived). Keep
                         # current_code_parity — IR infeasible (see MODE NOTE);
                         # realistic fills come from the fast_realism re-score.
# Phase 10 default-on: trial 0 is pinned to the actual-contract
# parameter set (the live config) so the optimizer's best can be
# compared against the live incumbent. Set False to disable.
INCUMBENT_TRIAL = True
# Gate the slow Top-N robustness sweep (cell below): True = run it (default,
# interactive). The papermill smoke test injects False to keep execution fast.
RUN_TOPN_SWEEP = True
# Two-stage finalist validation (§10i). The Top-N sweep cell below carries the
# top N_FINALISTS (ranked by median fold score − variance penalty) into the
# robustness + holdout evaluation and — when RESCORE_MODE is set — RE-SCORES them
# under that mode (fast_realism = honest depth/participation fills) instead of the
# cheap search mode. This turns a thorough current_code_parity search into a
# deploy decision made on realistic execution. Set RESCORE_MODE=None to re-score
# under the search mode (single-stage).
N_FINALISTS = 64         # finalists carried into the robustness + re-score stage
RESCORE_MODE = 'fast_realism'  # re-score finalists under this mode; None -> search mode


# 10 - Walk-Forward Optuna

Runs a **real** walk-forward parameter optimization against the shared
market-data lake. Each Optuna trial samples a parameter set, applies it
to the config, and runs a real backtest over every walk-forward
validation window; the trial objective is the median fold score. The
final-holdout window is never read during tuning.

**Feed auto-selection (`FEED='auto'`):** the notebook probes the lake
and adapts the run to the best data available -

| Lake holds | feed | simulation.mode |
|---|---|---|
| SIP bars + SIP quotes | `sip` | `intended_realism` |
| SIP bars, no SIP quotes | `sip` | `current_code_parity` |
| IEX bars only | `iex` | `current_code_parity` |
| neither | - | `smoke_fixture` (synthetic) |

Set `FEED` to `sip` / `iex` / `synthetic` to override the probe.

**Parameters:** `N_TRIALS` is the total trial count (`None` -> the
config's `optuna.n_trials`). `N_STARTUP_TRIALS` is how many of those are
random-sampling trials before TPE-guided search begins.

**Compute:** a run is `N_TRIALS` x `n_folds` real backtests - the config
default is a multi-day job. Set a small `N_TRIALS` for a quick run.

In [3]:
import time
start_time = time.perf_counter()

# Probe the lake and adapt the config's feed + simulation.mode.
# MODE_OVERRIDE pins simulation.mode over the feed-derived one (see the params
# cell): auto/sip would pick intended_realism now that quotes exist, but IR is
# infeasible on this lake (no halt data), so we keep current_code_parity.
from bowaka_v2_lab.optuna.autoconfig import resolve_walkforward_config
resolved = resolve_walkforward_config(
    CONFIG_PATH, feed_override=FEED, mode_override=MODE_OVERRIDE)
print(f'feed   : {resolved.feed}')
print(f'mode   : {resolved.mode}')
print(f'reason : {resolved.reason}')
print(f'config : {resolved.path}')


feed   : sip
mode   : current_code_parity
reason : SIP bars and SIP quotes present — full intended_realism run; mode_override=current_code_parity (was intended_realism)
config : research_notebooks/bowaka_v2_lab/artifacts/resolved_configs/walkforward_resolved.yml


In [4]:
import json
from bowaka_v2_lab.optuna.walkforward_runner import run_walkforward_study
# Realism remediation 2 Phase 8 (audit §P0-011): current_code_parity
# studies require an explicit opt-in. ResolvedWalkforwardConfig
# carries the auto-opt-in flags when the lake forces parity mode
# (IEX-only / SIP-bars-no-quotes); the mechanical cap is research_only.
#
# skip_best_trial_report=True: skip the study's internal single-best neighbour
# sweep (it re-runs folds in full artifact mode -> DQ not cached -> slow, and the
# holdout it would score needs a holdout-scope matrix that isn't built). The
# Top-N robustness sweep cell below does the finalist evaluation + report + YAMLs.
result = run_walkforward_study(
  resolved.path, n_trials=N_TRIALS, n_jobs=N_JOBS,
  n_startup_trials=N_STARTUP_TRIALS, allow_smoke=resolved.allow_smoke,
  allow_current_code_parity_study=resolved.allow_current_code_parity_study,
  tier=resolved.tier,
  incumbent_trial=INCUMBENT_TRIAL,
  skip_best_trial_report=True,
)
print(json.dumps(result, indent=2, default=str))

2026-06-10 07:25:04,655 INFO preflight passed: 4 checks
2026-06-10 07:43:28,123 INFO full per-fold preflight passed (mode=current_code_parity): 4 folds, 16 checks, 0 partial-tape limitation(s): []
/quants-lab/research_notebooks/bowaka_v2_lab/src/bowaka_v2_lab/optuna/dispatcher.py:78: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler=optuna.samplers.TPESampler(
[I 2026-06-10 07:43:28,905] A new study created in RDB with name: bowaka_v2_sip_walkforward_conservative_c4068194_20260610
2026-06-10 07:43:29,028 INFO walk-forward study bowaka_v2_sip_walkforward_conservative_c4068194_20260610: 5000 trials (500 random startup) x 3 folds, feed=sip, search_space_version=3; per-fold universe = daily point-in-time set (~1903 eligible on 2025-08-27) (preflight probe sampled 100 symbols)
2026-06-10 07:43:29,267 INFO incumbent baseline: enqueued trial 0 with 30 params (all from the mapped actual contract; no padding)
2026-06-10 

{
  "status": "ok",
  "study_name": "bowaka_v2_sip_walkforward_conservative_c4068194_20260610",
  "simulation_mode": "current_code_parity",
  "simulation_contract": "current_code_parity",
  "suitability_tier": "research_only",
  "feed": "sip",
  "partial_tape": false,
  "search_space_version": 3,
  "n_trials_requested": 5000,
  "n_trials_completed": 4881,
  "n_startup_trials": 500,
  "n_folds": 3,
  "universe": {
    "selection": "daily_point_in_time",
    "note": "each fold rebuilds the per-session PIT universe from the lake (exchange / price-band / ADV / exclusion / blocklist filters) \u2014 the trading universe is NOT capped and changes daily, like the live scanner",
    "preflight_probe_symbols": 100,
    "pit_sample": {
      "session": "2025-08-27",
      "eligible_symbols": 1903
    },
    "preflight_symbol_count": 100,
    "pit_union_symbol_count": 3130,
    "preflight_coverage_fraction": 0.01437699680511182,
    "research_waiver_capped_symbols": false
  },
  "final_holdout": [

# Top-N finalist robustness + holdout sweep (two-stage §10i)

The study above optimises the **objective** (v2 = log-return **minus** edge / turnover / fill penalties, so it can be negative even when PnL is positive). The single best trial by objective is not necessarily the best config to ship — a high score can sit on a *fragile spike* that collapses under a small parameter change, and out-of-sample behaviour matters more than the dev score.

This cell carries the **top `N_FINALISTS`** (parameters cell; default **64**) from the search study into a side-by-side finalist evaluation:

- **Two-stage re-score (`RESCORE_MODE`)** — finalists are *selected* from the cheap `current_code_parity` search but *re-scored* under **`fast_realism`** (honest depth/participation fills), so the deploy decision is made on realistic execution rather than legacy fills. Set `RESCORE_MODE=None` to re-score under the search mode (single-stage). The scan matrix is reused — `simulation.mode` is not in its hash.
- **Neighbour / robustness sweep** — for each finalist, re-score `NEIGHBOURS` parameter sets perturbed ±10% on the validation folds (flat plateau vs fragile spike).
- **Final-holdout** — score each finalist on the reserved holdout window (skipped gracefully if its scan-matrix isn't built).
- **PnL + full quant metrics** — net return, max drawdown, worst-day loss, win rate, avg/median trade return, # trades, fill rate, per-fold detail.
- Writes a **markdown comparison report** (rendered inline below) + a JSON sidecar + **two single-best YAMLs**: the robustness winner and the study's #1 by objective.

It runs `scripts/topn_robustness_sweep.py` as a subprocess (fork-parallel is unsafe inside a Jupyter kernel). Tune `N_FINALISTS` (parameters cell) / `NEIGHBOURS` / `JOBS`. Budget scales with `N_FINALISTS` × `NEIGHBOURS` re-scores — building the fold contexts once dominates, and under `fast_realism` the per-finalist backtests are heavier than under the cheap search mode.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from IPython.display import Markdown, display

# --- knobs ------------------------------------------------------------------
TOP_N = N_FINALISTS  # finalists to evaluate (set in the parameters cell; default 64)
NEIGHBOURS = 7       # ±10% perturbation neighbours per finalist
JOBS = None          # parallel workers; None -> min(TOP_N, cpu-2)
# Which study: None -> auto-detect the most recent walk-forward study in storage
# (the one cell 4 just produced). Set a name to target a specific finished study.
STUDY_NAME = None

_script = "research_notebooks/bowaka_v2_lab/scripts/topn_robustness_sweep.py"
_out = "research_notebooks/bowaka_v2_lab/artifacts/optuna/topn_robustness.md"
# Stage 1 (SELECTION): the RESOLVED config the study ran against (same plan / lake
# / lineage) — used to pick + rank the finalists from that study.
_cfg = resolved.path  # noqa: F821 (resolved is defined in the lake-probe cell above)

# Stage 2 (RE-SCORE): when RESCORE_MODE is set, derive a config identical to the
# search config except simulation.mode=RESCORE_MODE (fast_realism), so the top
# N_FINALISTS are re-scored under realistic depth/participation fills (the §10i
# two-stage). The scan matrix is reused — simulation.mode is not in the matrix
# hash. RESCORE_MODE=None -> single-stage (re-score under the search mode).
_rescore_cfg = None
if RUN_TOPN_SWEEP and RESCORE_MODE:
    from bowaka_v2_lab.optuna.autoconfig import derive_validation_config
    _rescore_cfg = "research_notebooks/bowaka_v2_lab/artifacts/resolved_configs/walkforward_rescore.yml"
    derive_validation_config(_cfg, validation_mode=RESCORE_MODE, out_path=_rescore_cfg)
    print(f"two-stage: top-{TOP_N} finalists selected from the {resolved.mode} study, "
          f"re-scored under {RESCORE_MODE}\n  rescore config: {_rescore_cfg}\n", flush=True)

if RUN_TOPN_SWEEP:
    _cmd = [sys.executable, _script, "--config", str(_cfg),
            "--top-n", str(TOP_N), "--neighbours", str(NEIGHBOURS), "--out", _out]
    if _rescore_cfg:
        _cmd += ["--rescore-config", _rescore_cfg]
    if JOBS:
        _cmd += ["--jobs", str(JOBS)]
    if STUDY_NAME:
        _cmd += ["--study-name", STUDY_NAME]

    # The script imports the lab from the working tree; pass PYTHONPATH to the child.
    _env = dict(os.environ)
    _pp = os.pathsep.join(["research_notebooks/bowaka_v2_lab/src",
                           "research_notebooks/bowaka_common/src"])
    _env["PYTHONPATH"] = _pp + (os.pathsep + _env["PYTHONPATH"] if _env.get("PYTHONPATH") else "")

    print("Top-N robustness + holdout sweep — builds fold contexts once, then sweeps finalists in parallel.")
    print("(progress streams below.)\n", flush=True)
    _proc = subprocess.Popen(_cmd, cwd=os.getcwd(), env=_env,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1)
    for _line in _proc.stdout:
        print(_line, end="")
    _proc.wait()

    if _proc.returncode == 0 and Path(_out).exists():
        print("\n" + "=" * 72 + "\nReport (also at " + _out + "):\n")
        display(Markdown(Path(_out).read_text(encoding="utf-8")))
        for _y in sorted(Path(_out).parent.glob(Path(_out).stem + "_*.yml")):
            print("exported config YAML:", _y)
    else:
        print(f"\nsweep exited with code {_proc.returncode} — see the output above.")
else:
    print('Top-N robustness sweep skipped (RUN_TOPN_SWEEP=False; e.g. the papermill smoke test).')


In [6]:
end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Script finished in {execution_time:.4f} seconds.")

execution_minutes = execution_time / 60
print(f"Script finished in {execution_minutes:.2f} minutes.")

execution_hours = execution_time / 3600
print(f"Script finished in {execution_hours:.2f} hours.")

Script finished in 76450.6115 seconds.
Script finished in 1274.18 minutes.
Script finished in 21.24 hours.
